# 04 · Tools

> **Kernel:** `Phase 5 · OpenAI Agents SDK` · **Needs:** Ollama.

Tools are how an agent acts on the world. The SDK makes defining them almost
trivial, but the details decide whether an agent is reliable: what the model
sees, what happens when a tool fails or hangs, what happens *after* a tool runs,
and which tools are available to whom.

**You will learn**
- `@function_tool`: how the name, docstring and type hints become the schema
- Tool errors: the SDK's default (report to the model) vs raising, and when to use each
- **Timeouts** on tools
- `tool_use_behavior`: letting a tool's result *be* the final answer
- **Conditional tools** with `is_enabled` (e.g. admin-only tools)
- **Agents as tools**: calling a specialist agent like a function
- OpenAI's hosted tools, and when you can use them

**Prerequisites:** notebooks 01–02 (and context, notebook 02 §6).

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 02_OPENAI_AGENTS_SDK folder
sys.path.insert(0, str(SECTION))

import asyncio, json
from typing import Literal
from agents import Agent, Runner, function_tool, RunContextWrapper, StopAtTools, UserError
from common import get_model

model = get_model()


def show(result):
    for item in result.new_items:
        kind = type(item).__name__
        if kind == 'ToolCallItem':
            print(f'  → {item.raw_item.name}({item.raw_item.arguments})')
        elif kind == 'ToolCallOutputItem':
            print(f'  ← {str(item.output)[:110]}')
    print('  final:', str(result.final_output)[:160])

## 1 · Why tool design matters

The model can't see your code. When it decides whether and how to call a tool,
it reads only the **schema**: a name, a description, and a JSON description of
the arguments. So a tool is really two things: the Python function your program
runs, and the **interface text** the model reads. Most tool-related agent bugs
come from the second.

```
 your code:  @function_tool                       what the model sees:
             def convert(amount: float,     ───►   name: "convert"
                         currency: Literal[...])   description: "Convert EUR to …"
                 '''Convert EUR to ...'''           parameters: {amount: number,
                                                                 currency: enum[USD, INR]}
```

## 2 · Defining tools

The **function name** becomes the tool name, the **docstring** the description,
and the **type hints** the parameter schema. The SDK also parses the `Args:`
section of the docstring, so each argument gets its own description:

In [ ]:
@function_tool
def convert(amount: float, currency: Literal['USD', 'INR']) -> str:
    """Convert an amount in EUR to another currency at today's rate.

    Args:
        amount: the amount in euros
        currency: the currency to convert to
    """
    rate = {'USD': 1.08, 'INR': 90.5}[currency]
    return f'{amount} EUR = {amount * rate:.2f} {currency}'


print('name       :', convert.name)
print('description:', convert.description)
print(json.dumps(convert.params_json_schema, indent=2))

`Literal['USD', 'INR']` became an **enum**, so the model *can't* ask for
`'GBP'`. Precise types are the cheapest reliability improvement there is.
Pydantic models and dataclasses work as argument types too, and `async def`
tools are supported (and preferred for I/O).

**Good tool design, in short:**

| Do | Why |
|---|---|
| a clear verb name: `search_orders`, not `helper2` | the name is the model's first hint |
| say *when* to use it in the docstring | helps choose between similar tools |
| precise types (`Literal`, `int`, models) | the schema constrains arguments |
| short, informative return strings | results go back into the context window |
| one job per tool | small tools compose; big ones confuse |

## 3 · When a tool fails

What should happen when a tool raises? The SDK's **default** is to catch the
exception and send its message **to the model** as the tool result, so the model
can react (retry with other arguments, or explain to the user):

In [ ]:
@function_tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b

calc = Agent(name='Calculator', model=model, tools=[divide],
             instructions='Use the divide tool, then explain the result.')
show(await Runner.run(calc, 'Use the divide tool to compute 10 / 0.'))

The model received `An error occurred while running the tool… float division by
zero` and explained it. (Notice the prompt says *use the divide tool*. Asked
"what is 10 / 0?", the model often answers from its own knowledge without
calling any tool. Whether a tool gets used is always the model's decision.) **This default is the opposite of LangGraph's**, where
a tool exception stops the graph unless you opt in to error handling. Neither
is wrong:

- *Report to the model* keeps agents resilient to expected failures (bad input,
  not found, rate limits).
- *Raise* makes real bugs visible instead of letting the model paper over them.

You choose per tool. `failure_error_function=None` means "raise":

In [ ]:
@function_tool(failure_error_function=None)
def strict_divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b

try:
    await Runner.run(calc.clone(tools=[strict_divide]), 'Use the divide tool to compute 10 / 0.')
except UserError as err:
    print('the run stopped:', err)

Or pass your own function to write a better message for the model (for example
one that doesn't leak internal details):

In [ ]:
def friendly_error(ctx: RunContextWrapper, error: Exception) -> str:
    return f'The calculation failed ({type(error).__name__}). Tell the user the input is invalid.'

@function_tool(failure_error_function=friendly_error)
def safe_divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b

show(await Runner.run(calc.clone(tools=[safe_divide]), 'Use the divide tool to compute 10 / 0.'))

## 4 · Timeouts

A tool that hangs stalls the whole run. `@function_tool(timeout=seconds)` cancels
it. By default (`timeout_behavior='error_as_result'`) the model is told the tool
timed out, so it can carry on without it:

In [ ]:
@function_tool(timeout=0.5)
async def inventory_api(sku: str) -> str:
    """Check stock for a product SKU in the warehouse system."""
    await asyncio.sleep(5)                 # pretend: the warehouse API hangs
    return f'{sku}: 12 in stock'

stock = Agent(name='Stock', model=model, tools=[inventory_api],
              instructions='Check stock with the tool. If it fails, say so honestly.')
show(await Runner.run(stock, 'How many C2 robots (SKU C2-STD) are in stock?'))

Timeouts need an `async` tool, since only coroutines can be cancelled cleanly.
For sync code, set a timeout on the client you call inside the tool.

## 5 · What happens after a tool runs: `tool_use_behavior`

By default (`'run_llm_again'`) tool results go back to the model, which then
writes the answer. Sometimes that's wasteful or even harmful: the tool's output
**is** the answer, and letting the model paraphrase it risks changing a price or
a legal text. `tool_use_behavior='stop_on_first_tool'` ends the run with the
tool's output as the final output:

In [ ]:
@function_tool
def official_price(product: str) -> str:
    """The official price of a product, exactly as it must be quoted."""
    return f'{product}: EUR 1,299.00 incl. VAT (price list 2026-09)'

quoter = Agent(name='Quoter', model=model, tools=[official_price],
               instructions='Look up prices with the tool.', tool_use_behavior='stop_on_first_tool')
result = await Runner.run(quoter, 'How much is the robot kit?')
print(repr(result.final_output))
print('model calls:', len(result.raw_responses))

One model call instead of two, and the answer is exactly the tool's text. The
other options:

| `tool_use_behavior=` | Effect |
|---|---|
| `'run_llm_again'` (default) | results go back to the model, which continues |
| `'stop_on_first_tool'` | the first tool's output is the final output |
| `StopAtTools(stop_at_tool_names=['book'])` | stop only if one of these tools was called |
| a function `(ctx, tool_results) -> ToolsToFinalOutputResult` | decide in code |

## 6 · Conditional tools: `is_enabled`

Not every user should have every tool. `is_enabled` can be a function of the
run's context. When it returns `False`, the tool is **removed from the schema
entirely**, so the model never knows it exists:

In [ ]:
def is_admin(ctx: RunContextWrapper, agent: Agent) -> bool:
    return bool(ctx.context and ctx.context.get('role') == 'admin')

@function_tool(is_enabled=is_admin)
def delete_account(username: str) -> str:
    """Permanently delete a user account."""
    return f'account {username!r} deleted'

support = Agent(name='Support', model=model, tools=[delete_account],
                instructions='Help with account requests. Use tools when available.')

for role in ('customer', 'admin'):
    print(f'as {role}:')
    show(await Runner.run(support, 'Delete the account of user bob.', context={'role': role}))

As a customer the model had no delete tool, so it couldn't be talked into using
it, however the prompt was phrased. **Permissions enforced by removing tools are
far stronger than permissions described in instructions.**

## 7 · Agents as tools

A whole agent can be a tool: `agent.as_tool(...)` wraps it so another agent can
call it with an input string and get back its final output. The calling agent
**stays in control**. It's like calling an expert on the phone and then
continuing the conversation yourself. (Compare with **handoffs** in notebook 06,
where control moves to the other agent.)

In [ ]:
translator = Agent(name='Translator', model=model,
                   instructions='Translate the text into Dutch. Reply with the translation only.')
editor = Agent(name='Editor', model=model,
               instructions='Rewrite the text to be shorter and clearer. Reply with the text only.')

writer = Agent(
    name='Comms lead', model=model,
    instructions=('You prepare customer announcements. First make the draft shorter with the editor, '
                  'then translate the edited version with the translator. Return both versions.'),
    tools=[editor.as_tool(tool_name='edit_text', tool_description='Make a text shorter and clearer.'),
           translator.as_tool(tool_name='translate_to_dutch', tool_description='Translate text into Dutch.')],
)
show(await Runner.run(writer, 'Draft: "We would like to inform all of our valued customers that the robot '
                              'delivery service will unfortunately be paused on Monday for maintenance work."'))

Each specialist ran its own complete loop (its own instructions, model calls,
even its own tools) and returned just its final output to the Comms lead.

**Check the order of the calls.** The instructions said *first* edit, *then*
translate the edited version. In our runs the model often requested **both tools
in the same turn**. They then run in parallel, so the translator received the
**original** draft, not the edited one. A prompt can ask for an order, but when
the model batches calls there's nothing to enforce it. If step B needs step A's
output, sequence them **in code** (notebook 05): `edited = await Runner.run(editor, …)`,
then translate `edited`.

## 8 · Hosted tools (OpenAI only)

With OpenAI models on the Responses API, the SDK also offers **hosted tools**
that run on OpenAI's servers: `WebSearchTool`, `FileSearchTool` (search your
uploaded documents), `CodeInterpreterTool`, `ImageGenerationTool` and more. They
need no code on your side, but they only work with OpenAI's Responses models, so
they aren't available on Ollama. For local agents you write the equivalent as
function tools or use MCP servers (notebook 11).

## 9 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Vague names / docstrings | the model picks the wrong tool or none | verb names, "use when…" docstrings |
| `str` where a fixed set is meant | invalid arguments | `Literal[...]` |
| Letting the model reword authoritative data | changed prices or legal text | `stop_on_first_tool` / `StopAtTools` |
| Permissions only in instructions | the model is talked into using a tool | `is_enabled` from context |
| Sync tools that hang | the whole run stalls | `async` tools with `timeout=` |
| Swallowing real bugs as "tool errors" | the model hides a broken tool | `failure_error_function=None` for code that must not fail |
| Huge tool outputs | context and cost blow up | return only what the model needs |
| Relying on the prompt to order dependent tool calls | calls run in parallel on stale input | sequence dependent steps in code |

## 10 · Check yourself

<details><summary><b>1.</b> What does the model actually see of a <code>@function_tool</code>?</summary>

Only the schema: the tool name (from the function name), the description (from the docstring), and the parameter schema (from type hints, with per-argument descriptions from the docstring's `Args:` section). Never the code.
</details>

<details><summary><b>2.</b> A tool raises an exception. What does the SDK do by default, and how do you make it stop the run instead?</summary>

By default it sends the error message to the model as the tool's result, and the run continues. `@function_tool(failure_error_function=None)` makes the exception propagate (as a `UserError`) and stop the run.
</details>

<details><summary><b>3.</b> When would you use <code>tool_use_behavior='stop_on_first_tool'</code>?</summary>

When the tool's output is the answer and must not be reworded (prices, legal text, computed results), or when a second model call would just waste time and tokens.
</details>

<details><summary><b>4.</b> Why is <code>is_enabled</code> stronger than telling the agent "only admins may delete accounts"?</summary>

Disabled tools are removed from the schema, so the model can't call them regardless of what the prompt says. An instruction is only a request that clever or malicious input can talk the model out of.
</details>

<details><summary><b>5.</b> Agent-as-tool vs handoff: who has control afterwards?</summary>

With an agent-as-tool, the calling agent keeps control and receives the sub-agent's output as a tool result. With a handoff, control of the conversation moves to the other agent (notebook 06).
</details>

## Takeaway

A tool is code plus the interface the model reads, so write names, docstrings
and types for the model. Decide per tool whether failures are reported to the
model (default) or raised; put timeouts on async tools; use `tool_use_behavior`
when a tool's output *is* the answer; gate tools with `is_enabled`; and wrap
specialist agents with `as_tool` when the caller should stay in charge.

Next → `05_orchestrating_in_code.ipynb`: chaining, routing, parallelising and
looping agents with plain Python.